# 01 - Fundamentos de Agrupamiento Basado en Densidad (DBSCAN)

## Curso de Machine Learning: Modelos No Supervisados

En el modulo anterior estudiamos **K-Means**, un algoritmo de particionamiento geometrico sumamente eficiente pero gobernado por tres supuestos restrictivos: asume que todos los clusters son **convexos (esfericos)**, poseen **varianzas isotropicas similares** y obliga a que **absolutamente todas las observaciones pertenezcan a algun cluster**, incluso si se trata de errores de medicion o eventos extremos.

En el analisis de datos financieros, bursatiles y transaccionales, estos supuestos se rompen sistematicamente. Los mercados financieros exhiben distribuciones de colas pesadas (*fat tails*), relaciones no lineales complejas entre riesgo y rendimiento, y **eventos anomalos extremos** (*flash crashes*, empresas en *distress* financiero, fraude corporativo o *insider trading*). Cuando aplicamos K-Means sobre datos financieros con valores atipicos, los *outliers* ejercen una traccion gravitacional cuadratica sobre los centroides ($\mu_k$), distorsionando la segmentacion de todo el portafolio.

En este cuaderno estudiaremos **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*), propuesto por Ester et al. (1996), el algoritmo estandar de la industria para descubrir grupos de **forma geometrica arbitraria** mientras **aisla automaticamente el ruido y las anomalias**.

---

### Contenidos
1. Motivacion en Finanzas Cuantitativas: Limites de K-Means ante Variedades No Lineales y Cisnes Negros
2. Intuicion Topologica: Agrupamiento por Regiones de Alta Densidad Continua
3. Conceptos Formales: $\varepsilon$-Vecindad, Puntos Nucleo (*Core*), Puntos Frontera (*Border*) y Ruido (*Noise*)
4. Relaciones de Densidad: Alcanzabilidad Directa, Alcanzabilidad por Densidad y Conectividad
5. El Algoritmo DBSCAN Paso a Paso y Complejidad Computacional ($O(n \log n)$ con KD-Trees)
6. Demostracion Visual en 2D: Anatomia de Puntos y Superioridad Topologica frente a K-Means
7. Caso de Estudio Financiero: Microestructura de Mercado y Regimenes de Riesgo de Credito Corporativo
8. Preprocesamiento Financiero: Por que usar `RobustScaler` en lugar de `StandardScaler`
9. Comparativa Empirica: K-Means (Contaminado por Outliers) vs. DBSCAN
10. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. Motivacion en Finanzas Cuantitativas: Limites de K-Means

En Ingenieria Financiera y Gestion de Riesgos (*Risk Management*), los activos rara vez se agrupan en nubes esfericas perfectas:
- **No linealidad estructural**: La relacion entre el apalancamiento corporativo (Deuda/EBITDA) y el costo de cobertura crediticia (*Credit Default Swap Spread*) sigue curvas convexas exponenciales, no hiperplanos lineales.
- **Vulnerabilidad de la media aritmetica**: K-Means define cada centroide como la media $\mu_k = \frac{1}{|C_k|}\sum x_i$. En estadistica robusta sabemos que el punto de ruptura (*breakdown point*) de la media aritmetica es $0\%$: **basta una sola observacion extrema** (ej. un bono corporativo en impago con un spread de $4,500$ bps) para desplazar severamente el centroide de su cluster.
- **Ausencia de concepto de anomalia en K-Means**: K-Means particiona todo el espacio $\mathbb{R}^p$ en celdas de Voronoi infinitas. Un evento fraudulento situado a 20 desviaciones estandar del centro seguira siendo etiquetado como miembro legitimo de algun cluster.

**DBSCAN** cambia radicalmente el paradigma: en lugar de buscar centros de masa esfericos, define los clusters como **regiones maximas de alta densidad conectadas**, separadas entre si por regiones vacias o de baja densidad (donde habitan las anomalias financieras).

## 2. Conceptos Formales: $\varepsilon$-Vecindad y Clasificacion de Puntos

El algoritmo DBSCAN esta completamente parametrizado por dos hiperparametros intuitivos:
1. **$\varepsilon$ (`eps`)**: El radio maximo de vecindad alrededor de una observacion.
2. **$MinPts$ (`min_samples`)**: El numero minimo de observaciones requeridas dentro del radio $\varepsilon$ para considerar que una region es "densa".

### Definicion 1: $\varepsilon$-Vecindad ($N_\varepsilon(p)$)
Dado un punto $p \in X$, su $\varepsilon$-vecindad es el conjunto de todas las observaciones del dataset cuya distancia a $p$ es menor o igual a $\varepsilon$ (incluyendo al propio punto $p$):
$$N_\varepsilon(p) = \{ q \in X \mid \text{dist}(p, q) \leq \varepsilon \}$$

A partir del cardinal $|N_\varepsilon(p)|$, DBSCAN clasifica cada observacion del universo de datos en exactamente una de **tres categorias mutuamente excluyentes**:

1. **Punto Nucleo (*Core Point*)**: Un punto $p$ es nucleo si su vecindad contiene al menos $MinPts$ observaciones:
   $$|N_\varepsilon(p)| \geq MinPts$$
   *Interpretacion financiera*: Representa un activo o transaccion situados en el corazon denso de un regimen de mercado normal.

2. **Punto Frontera (*Border Point*)**: Un punto $q$ es frontera si **no** alcanza el umbral de densidad por si mismo ($|N_\varepsilon(q)| < MinPts$), pero **cae dentro de la $\varepsilon$-vecindad de al menos un Punto Nucleo** $p$ ($q \in N_\varepsilon(p)$).
   *Interpretacion financiera*: Representa un activo en la periferia o transicion de un regimen (mayor volatilidad relativa, pero aun conectado al comportamiento del grupo).

3. **Punto de Ruido o Anomalia (*Noise / Outlier Point*)**: Un punto $r$ que no es Punto Nucleo ni cae en la $\varepsilon$-vecindad de ningun Punto Nucleo. Scikit-Learn le asigna automaticamente la etiqueta especial **`-1`**.
   *Interpretacion financiera*: Eventos aislados de cola (*distressed debt*, *flash crashes*, anomalias operativas o fraude).

## 3. Relaciones de Densidad y Formacion de Clusters

¿Como une DBSCAN millones de esferas locales de radio $\varepsilon$ para formar estructuras curvas de cualquier topologia? Mediante tres definiciones de conectividad:

1. **Directamente Alcanzable por Densidad**: Un punto $q$ es directamente alcanzable por densidad desde $p$ si:
   - $p$ es un **Punto Nucleo** ($|N_\varepsilon(p)| \geq MinPts$), y
   - $q \in N_\varepsilon(p)$.
   *(Nota: Esta relacion no es simetrica si $q$ es un Punto Frontera).*

2. **Alcanzable por Densidad (*Density-Reachable*)**: Un punto $q$ es alcanzable por densidad desde $p$ si existe una cadena de puntos $p_1, p_2, \dots, p_m$ (donde $p_1 = p$ y $p_m = q$) tal que cada $p_{i+1}$ es directamente alcanzable por densidad desde $p_i$. Todos los eslabones intermedios $p_1, \dots, p_{m-1}$ deben ser **Puntos Nucleo**.

3. **Conectado por Densidad (*Density-Connected*)**: Dos puntos $p$ y $q$ estan conectados por densidad si existe un Punto Nucleo $o$ desde el cual tanto $p$ como $q$ son alcanzables por densidad.

> **Definicion Formal de Cluster en DBSCAN**: Un cluster $C$ es un subconjunto no vacio maximal de puntos mutuamente conectados por densidad. Si $p \in C$ y $p$ es un Punto Nucleo, entonces cualquier punto $q$ alcanzable por densidad desde $p$ tambien pertenece a $C$.

## 4. Demostracion Visual: Anatomia de Puntos Nucleo, Frontera y Ruido

Construyamos una demostracion grafica que ilustre como DBSCAN distingue internamente entre **Puntos Nucleo**, **Puntos Frontera** y **Ruido (`-1`)** utilizando los atributos `model.core_sample_indices_` y `model.labels_` de Scikit-Learn, y comparemos su capacidad para seguir variedades curvas frente a K-Means.

In [ ]:
# Generacion de una variedad no lineal con anomalias inyectadasX_moons, _ = make_moons(n_samples=350, noise=0.08, random_state=42)rng = np.random.default_rng(42)outliers = rng.uniform(low=[-1.5, -1.0], high=[2.5, 1.5], size=(25, 2))X_demo = np.vstack([X_moons, outliers])X_demo = StandardScaler().fit_transform(X_demo)# Ajuste de K-Means vs DBSCANkm_demo = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_demo)db_demo = DBSCAN(eps=0.32, min_samples=6).fit(X_demo)# Identificacion de tipos de puntos en DBSCANcore_mask = np.zeros_like(db_demo.labels_, dtype=bool)core_mask[db_demo.core_sample_indices_] = Truenoise_mask = (db_demo.labels_ == -1)border_mask = ~(core_mask | noise_mask)fig, axes = plt.subplots(1, 2, figsize=(15, 6))# Panel 1: K-Meansaxes[0].scatter(X_demo[:, 0], X_demo[:, 1], c=km_demo.labels_, cmap='coolwarm', s=40, edgecolor='k', alpha=0.8)axes[0].scatter(km_demo.cluster_centers_[:, 0], km_demo.cluster_centers_[:, 1],                 c='yellow', marker='X', s=250, edgecolor='k', lw=2, label='Centroides K-Means')axes[0].set_title('K-Means (K=2): Corte Lineal Erroneo y Absorcion de Ruido', fontsize=12, fontweight='bold')axes[0].set_xlabel('$X_1$ (Estandarizado)')axes[0].set_ylabel('$X_2$ (Estandarizado)')axes[0].legend()# Panel 2: DBSCAN desglosado por tipo de puntoaxes[1].scatter(X_demo[core_mask, 0], X_demo[core_mask, 1], c=db_demo.labels_[core_mask],                 cmap='viridis', s=65, edgecolor='k', alpha=0.9, label=f'Puntos Nucleo (n={core_mask.sum()})')axes[1].scatter(X_demo[border_mask, 0], X_demo[border_mask, 1], c=db_demo.labels_[border_mask],                 cmap='viridis', marker='^', s=60, edgecolor='k', alpha=0.9, label=f'Puntos Frontera (n={border_mask.sum()})')axes[1].scatter(X_demo[noise_mask, 0], X_demo[noise_mask, 1], c='#d62728',                 marker='x', s=70, lw=2, label=f'Ruido / Anomalias [-1] (n={noise_mask.sum()})')axes[1].set_title('DBSCAN (eps=0.32, min_samples=6): Topologia Exacta y Aislamiento de Ruido', fontsize=12, fontweight='bold')axes[1].set_xlabel('$X_1$ (Estandarizado)')axes[1].set_ylabel('$X_2$ (Estandarizado)')axes[1].legend(loc='upper right', frameon=True)plt.suptitle('Paradigma de Particionamiento (K-Means) vs. Paradigma de Densidad (DBSCAN)', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

## 5. Terminologia Formal de DBSCAN
| Termino | Definicion |
| :--- | :--- |
| **`eps` ($\varepsilon$)** | Radio maximo de la hiperesfera de vecindad alrededor de cada observacion. Controla la resolucion espacial local. |
| **`min_samples` ($MinPts$)** | Numero minimo de puntos dentro de la $\varepsilon$-vecindad (incluyendo el propio punto) para declarar una zona como densa. |
| **Punto Nucleo (*Core Point*)** | Observacion que cumple $|N_\varepsilon(p)| \geq MinPts$. Sus indices se almacenan en `dbscan.core_sample_indices_`. |
| **Punto Frontera (*Border Point*)** | Observacion con $|N_\varepsilon(q)| < MinPts$ pero vecina de al menos un Punto Nucleo. Recibe la etiqueta del cluster pero no expande la busqueda. |
| **Ruido / Outlier (`label = -1`)** | Observacion aislada en zonas de baja densidad que no pertenece a ningun cluster. |
| **Naturaleza Transductiva** | DBSCAN agrupa los datos observados mediante conectividad local, por lo que en Scikit-Learn **no implementa el metodo `.predict()`** para datos futuros (veremos como resolver esto en MLOps en el Cuaderno 03). |

## 6. Caso de Estudio Financiero: Regimenes de Credito Corporativo y Liquidez Institucional

En la mesa de analisis cuantitativo de un fondo de renta fija institucional, deseamos segmentar un universo de **550 emisiones de bonos corporativos y acciones cotizadas** utilizando 6 indicadores fundamentales de riesgo crediticio y microestructura de mercado:

1. `deuda_neta_ebitda`: Ratio de apalancamiento Deuda Neta / EBITDA (veces).
2. `cobertura_intereses_icr`: Cobertura de gastos financieros EBIT / Intereses (veces).
3. `spread_cds_5y_bps`: Prima de riesgo de impago en Credit Default Swaps a 5 anos (puntos basicos, bps).
4. `volatilidad_implicita_30d`: Volatilidad anualizada de opciones ATM a 30 dias (%).
5. `bid_ask_spread_bps`: Costo de friccion de liquidez en el libro de ordenes (bps).
6. `retorno_roe_pct`: Rentabilidad sobre recursos propios ROE (%).

En el mercado existen **3 regimenes naturales de emisores** (*Investment Grade Defensivo*, *Crecimiento High-Tech Alta Liquidez* y *High-Yield Ciclico Apalancado*), ademas de un **5% de emisores en situacion de estres agudo, manipulacion contable o iliquidez severa (*Distressed / Default Outliers*)**.

In [ ]:
def generar_dataset_credito_institucional(seed=42):
    rng = np.random.default_rng(seed)

    # Regimen 0: Investment Grade Defensivo (Bajo apalancamiento, alta cobertura, bajo spread CDS) - n=220
    r0_deuda = rng.normal(1.4, 0.35, 220).clip(0.3, 2.5)
    r0_icr = rng.normal(11.5, 2.0, 220).clip(6.5, 18.0)
    r0_cds = 45 + 18 * (r0_deuda ** 1.5) + rng.normal(0, 8, 220)
    r0_vol = rng.normal(16.0, 2.5, 220).clip(10.0, 23.0)
    r0_bidask = rng.normal(4.2, 0.9, 220).clip(1.8, 7.5)
    r0_roe = rng.normal(14.5, 2.8, 220).clip(7.0, 22.0)
    regimen_0 = np.column_stack([r0_deuda, r0_icr, r0_cds, r0_vol, r0_bidask, r0_roe])

    # Regimen 1: High-Tech / Crecimiento (Deuda casi nula, altisimo ROE, volatilidad moderada-alta, muy liquido) - n=160
    r1_deuda = rng.normal(0.45, 0.25, 160).clip(0.0, 1.2)
    r1_icr = rng.normal(24.0, 3.5, 160).clip(15.0, 35.0)
    r1_cds = rng.normal(65.0, 12.0, 160).clip(35.0, 100.0)
    r1_vol = rng.normal(31.0, 3.8, 160).clip(22.0, 42.0)
    r1_bidask = rng.normal(2.5, 0.6, 160).clip(1.0, 4.5)
    r1_roe = rng.normal(29.5, 4.2, 160).clip(20.0, 42.0)
    regimen_1 = np.column_stack([r1_deuda, r1_icr, r1_cds, r1_vol, r1_bidask, r1_roe])

    # Regimen 2: High-Yield Ciclico Apalancado (Deuda elevada, baja cobertura, alto CDS y mayor spread bid-ask) - n=140
    r2_deuda = rng.normal(4.3, 0.55, 140).clip(3.0, 5.8)
    r2_icr = rng.normal(2.8, 0.6, 140).clip(1.4, 4.5)
    r2_cds = 260 + 25 * (r2_deuda ** 1.4) + rng.normal(0, 25, 140)
    r2_vol = rng.normal(42.0, 4.5, 140).clip(32.0, 54.0)
    r2_bidask = rng.normal(14.5, 2.2, 140).clip(9.0, 21.0)
    r2_roe = rng.normal(6.5, 3.0, 140).clip(-2.0, 14.0)
    regimen_2 = np.column_stack([r2_deuda, r2_icr, r2_cds, r2_vol, r2_bidask, r2_roe])

    # Anomalias Financieras / Distressed Debt / Flash Illiquidity Shocks (Etiqueta real = -1) - n=30
    out_deuda = rng.uniform(7.5, 14.0, 30)
    out_icr = rng.uniform(-3.5, 0.9, 30)
    out_cds = rng.uniform(1100.0, 3800.0, 30)
    out_vol = rng.uniform(70.0, 145.0, 30)
    out_bidask = rng.uniform(38.0, 95.0, 30)
    out_roe = rng.uniform(-45.0, -12.0, 30)
    anomalias = np.column_stack([out_deuda, out_icr, out_cds, out_vol, out_bidask, out_roe])

    X_total = np.vstack([regimen_0, regimen_1, regimen_2, anomalias])
    y_regimen_real = np.array([0]*220 + [1]*160 + [2]*140 + [-1]*30)

    perm = rng.permutation(len(X_total))
    columnas = [
        'deuda_neta_ebitda', 'cobertura_intereses_icr', 'spread_cds_5y_bps',
        'volatilidad_implicita_30d', 'bid_ask_spread_bps', 'retorno_roe_pct'
    ]
    tickers = [f'BOND_{i:03d}' for i in range(1, len(X_total) + 1)]
    df = pd.DataFrame(X_total[perm], index=tickers, columns=columnas)
    return df, y_regimen_real[perm]

df_credito, y_regimen_oculto = generar_dataset_credito_institucional()
print(f"Dimensiones del portafolio analizado: {df_credito.shape}")
print("\nEstadisticas descriptivas (notese el sesgo extremo entre el percentil 75% y el Maximo por activos Distressed):")
print(df_credito.describe().T[['mean', 'std', '50%', '75%', 'max']].round(2))


## 7. Preprocesamiento Financiero: `RobustScaler` frente a `StandardScaler`

Al igual que K-Means, DBSCAN calcula distancias en $\mathbb{R}^p$ y requiere que todas las variables esten en escalas comparables. Sin embargo, en presencia de anomalias financieras (`spread_cds_5y_bps` llega a $3,800$ bps cuando la mediana es $105$ bps), **`StandardScaler` sufre un problema grave**:

$$Z_j = \frac{x_j - \mu_j}{\sigma_j}$$

Como los *outliers* inflan artificialmente la desviacion estandar $\sigma_j$, al dividir entre una $\sigma_j$ gigantesca **todos los activos normales quedan aplastados y comprimidos en un rango diminuto**, perdiendo resolucion espacial entre regimenes.

La solucion profesional para DBSCAN en finanzas es utilizar **`RobustScaler`**, que centra y escala utilizando estadisticos de orden inmunes a valores extremos (la **Mediana** y el **Rango Intercuartilico $IQR = Q_{75} - Q_{25}$**):

$$X_{robust, j} = \frac{x_j - \text{Mediana}(x_j)}{Q_{75}(x_j) - Q_{25}(x_j)}$$

In [ ]:
# Transformacion con RobustScaler vs StandardScalerscaler_std = StandardScaler()X_std = scaler_std.fit_transform(df_credito)scaler_rob = RobustScaler()X_rob = scaler_rob.fit_transform(df_credito)# Aplicamos DBSCAN sobre los datos escalados robustamentedbscan_fin = DBSCAN(eps=0.85, min_samples=10)labels_dbscan = dbscan_fin.fit_predict(X_rob)# Aplicamos K-Means (forzado a K=3 regimenes principales)kmeans_fin = KMeans(n_clusters=3, n_init=15, random_state=42)labels_kmeans = kmeans_fin.fit_predict(X_rob)# Conteo de elementos por cluster en DBSCANconteo_db = pd.Series(labels_dbscan).value_counts().sort_index()print("Distribucion de emisiones descubiertas por DBSCAN:")for etiqueta, cantidad in conteo_db.items():    nombre = 'RUIDO / ANOMALIAS FINANCIERAS (Distressed)' if etiqueta == -1 else f'Regimen Crediticio {etiqueta}'    print(f"  Etiqueta {etiqueta:2d} -> {cantidad:3d} emisiones ({nombre})")ari_db = adjusted_rand_score(y_regimen_oculto, labels_dbscan)ari_km = adjusted_rand_score(y_regimen_oculto, labels_kmeans)print("-" * 70)print(f"Concordancia global contra la estructura real (ARI) - K-Means: {ari_km:.4f}")print(f"Concordancia global contra la estructura real (ARI) - DBSCAN:  {ari_db:.4f}")

## 8. Visualizacion Comparativa: Contaminacion de Centroides en K-Means vs. Aislamiento en DBSCANObservemos que hizo cada algoritmo en el plano financiero **Apalancamiento (`deuda_neta_ebitda`) vs. Riesgo de Impago (`spread_cds_5y_bps`)**:
- **K-Means ($K=3$)** gasto uno de sus 3 clusters exclusivamente para capturar a los 30 bonos *distressed* extremos, viendose forzado a fusionar erroneamente dos regimenes sanos completamente distintos (*Investment Grade* y *High-Tech*) en un solo grupo.
- **DBSCAN** identifico de forma autonoma los **3 regimenes legitimos de mercado** y marco con precision quirurgica las **30 emisiones en estres extremo como `-1` (Ruido / Alerta de Riesgo)**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))col_x, col_y = 'deuda_neta_ebitda', 'spread_cds_5y_bps'# Panel 1: K-Meansscatter1 = axes[0].scatter(df_credito[col_x], df_credito[col_y], c=labels_kmeans,                            cmap='tab10', s=45, edgecolor='k', alpha=0.8)axes[0].set_title(f'K-Means (K=3): Los Outliers Secuestran un Cluster\ny Fusionan Regimenes Sanos (ARI = {ari_km:.3f})',                   fontsize=11.5, fontweight='bold')axes[0].set_xlabel('Deuda Neta / EBITDA (veces)', fontsize=10, fontweight='bold')axes[0].set_ylabel('Spread CDS 5Y (bps)', fontsize=10, fontweight='bold')# Panel 2: DBSCANmask_normales = (labels_dbscan != -1)mask_anomalias = (labels_dbscan == -1)axes[1].scatter(df_credito.loc[mask_normales, col_x], df_credito.loc[mask_normales, col_y],                 c=labels_dbscan[mask_normales], cmap='viridis', s=50, edgecolor='k', alpha=0.85, label='Regimenes Normales (0, 1, 2)')axes[1].scatter(df_credito.loc[mask_anomalias, col_x], df_credito.loc[mask_anomalias, col_y],                 c='#d62728', marker='x', s=80, lw=2.2, label='Anomalias Distressed (Cluster -1)')axes[1].set_title(f'DBSCAN: Separa los 3 Regimenes Sanos y Aisla\nlos Bonos Distressed como Ruido (ARI = {ari_db:.3f})',                   fontsize=11.5, fontweight='bold')axes[1].set_xlabel('Deuda Neta / EBITDA (veces)', fontsize=10, fontweight='bold')axes[1].set_ylabel('Spread CDS 5Y (bps)', fontsize=10, fontweight='bold')axes[1].legend(loc='upper left', frameon=True, fontsize=10)plt.suptitle('Impacto de Eventos Financieros de Cola en K-Means vs. DBSCAN', fontsize=14, fontweight='bold', y=1.03)plt.tight_layout()plt.show()

## 9. Conclusiones

### Puntos Clave:
1. **No requiere especificar $K$ a priori**: DBSCAN descubre automaticamente el numero de clusters en funcion de la estructura de densidad del espacio.
2. **Inmunidad frente a Outliers y Deteccion de Anomalias Nativa**: Al clasificar las regiones de baja densidad como `-1`, evita que los valores extremos distorsionen los grupos principales, ofreciendo un detector de anomalias financieras integrado.
3. **Importancia de `RobustScaler`**: En datasets financieros donde se aplica DBSCAN, escalar mediante la mediana y el rango intercuartilico ($IQR$) preserva la densidad compacta de los activos normales sin ser aplastados por la varianza de los *outliers*.
4. **Sensibilidad a $\varepsilon$ y `min_samples`**: Si elegimos un $\varepsilon$ demasiado pequeno, casi todo el dataset sera etiquetado como ruido (`-1`); si elegimos un $\varepsilon$ demasiado grande, todos los regimenes se fusionaran en un unico mega-cluster.

En el siguiente cuaderno, **02_Hiperparametros_KDist_y_Metricas_Densidad.ipynb**, aprenderemos el metodo matematico exacto basado en graficos de **$k$-distancias de vecinos mas cercanos** para calibrar $\varepsilon$ y `min_samples` sin adivinar.